## Step 1: Setup Environment and Clone Repository

In [ ]:
# Check if we're running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Running in local environment")

import os
import sys
from pathlib import Path

# Set working directory
if IN_COLAB:
    os.chdir('/content')
    
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Clone the repository (replace with your actual repository URL)
REPO_URL = "https://github.com/your-username/Footaball-analysis.git"  # Update this with your repo URL
REPO_NAME = "Footaball-analysis"

# Remove existing directory if it exists
if os.path.exists(REPO_NAME):
    !rm -rf {REPO_NAME}
    print(f"Removed existing {REPO_NAME} directory")

# Clone the repository
!git clone {REPO_URL}
print(f"✅ Repository cloned successfully")

# Change to repository directory
os.chdir(REPO_NAME)
print(f"Changed to directory: {os.getcwd()}")

# List directory contents
!ls -la

## Step 2: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 libgoogle-perftools4

# Install Python packages
!pip install -q supervision==0.27.0
!pip install -q ultralytics>=8.3.235
!pip install -q transformers>=4.57.3
!pip install -q tokenizers==0.22.1
!pip install -q autoprocessor>=0.9.0
!pip install -q gdown>=5.2.0
!pip install -q roboflow>=1.2.11
!pip install -q python-dotenv>=1.2.1
!pip install -q tqdm>=4.67.1

print("✅ Dependencies installed successfully")

## Step 3: Setup Python Path and Imports

In [ ]:
# Test imports
try:
    import supervision as sv
    from ultralytics import YOLO
    import torch
    import umap  # Add umap import
    from sklearn.cluster import KMeans  # Add sklearn import
    from datetime import datetime
    from typing import Tuple
    import numpy as np
    
    print("✅ Basic imports successful")
    print(f"Supervision version: {sv.__version__}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        
except ImportError as e:
    print(f"❌ Import error: {e}")

In [ ]:
# Import project modules
try:
    from src.model import get_models
    from src.team import TeamClassifier, TeamConsistencyTracker
    from src.frame_processor import initialize_frame_processor, process_frame
    from src.annotators import get_annotators, get_tracker
    from src.config import (
        OUTPUT_DIR,
        PLAYER_DETECTION_MODEL_PATH,
        KEYPOINT_MODEL_PATH,
        BALL_ID,
        GOALKEEPER_ID,
        PLAYER_ID,
        REFEREE_ID,
    )
    
    print("✅ Project modules imported successfully")
    
except ImportError as e:
    print(f"❌ Project import error: {e}")
    print("Make sure the repository was cloned correctly and contains all source files")

## Step 4: Upload Video File and Check Models

In [ ]:
# Upload video file
if IN_COLAB:
    from google.colab import files
    
    print("Please upload your video file:")
    uploaded = files.upload()
    
    # Get the uploaded file name
    video_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {video_filename}")
    
    # Move to inputs directory
    inputs_dir = Path("inputs")
    inputs_dir.mkdir(exist_ok=True)
    
    import shutil
    video_path = inputs_dir / video_filename
    shutil.move(video_filename, video_path)
    
    print(f"✅ Video moved to: {video_path}")
else:
    # For local environment, specify your video path
    video_path = input("Please enter the path to your video file: ")
    video_path = Path(video_path)
    
if not video_path.exists():
    print(f"❌ Video file not found: {video_path}")
else:
    print(f"✅ Video file ready: {video_path}")
    INPUT_VIDEO_PATH = str(video_path)

In [ ]:
# Check if model files exist
models_dir = Path("models")
player_model_path = models_dir / "player-detection.pt"
keypoint_model_path = models_dir / "keypoint-detection.pt"

print("Checking model files:")
print(f"Models directory: {models_dir.exists()}")
print(f"Player detection model: {player_model_path.exists()}")
print(f"Keypoint detection model: {keypoint_model_path.exists()}")

if not player_model_path.exists() or not keypoint_model_path.exists():
    print("\n⚠️ Model files not found. You may need to:")
    print("1. Download the model files manually")
    print("2. Or use pre-trained YOLO models for testing")
    
    # Option to use default YOLO models
    use_default = input("\nUse default YOLO models for testing? (y/n): ").lower() == 'y'
    
    if use_default:
        print("Using default YOLO models...")
        # Will be handled in the model loading section
        USE_DEFAULT_MODELS = True
    else:
        print("Please upload your model files to the models/ directory")
        USE_DEFAULT_MODELS = False
else:
    print("✅ All model files found")
    USE_DEFAULT_MODELS = False

## Step 5: Initialize Models and Pipeline Components

In [ ]:
def create_output_directory() -> Path:
    """Create output directory with timestamp"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = Path(OUTPUT_DIR) / f"analysis_{timestamp}"
    output_path.mkdir(parents=True, exist_ok=True)
    return output_path

# Create output directory
output_dir = create_output_directory()
print(f"✅ Output directory created: {output_dir}")

In [ ]:
def setup_models_and_annotators() -> Tuple:
    """Initialize all models and annotators with stability improvements for Colab"""
    print("Loading models...")
    
    if USE_DEFAULT_MODELS:
        # Use default YOLO models for testing
        print("Using default YOLO models...")
        player_detection_model = YOLO("yolov8n.pt")
        keypoint_model = YOLO("yolov8n-pose.pt")
    else:
        # Load custom models
        player_detection_model, keypoint_model = get_models(
            player_detection_model_path=PLAYER_DETECTION_MODEL_PATH,
            keypoint_model_path=KEYPOINT_MODEL_PATH,
        )
    
    # Initialize team classifier with improved stability
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Use CPU for team classifier to improve stability in Colab
    if IN_COLAB:
        print("Using CPU for team classifier to improve stability in Colab")
        team_classifier = TeamClassifier(device="cpu", batch_size=16)  # Smaller batch size
    else:
        team_classifier = TeamClassifier(device=device)
    
    # Initialize tracker with reset
    tracker = get_tracker()
    tracker.reset()
    
    # Initialize team tracker with more conservative settings for Colab
    if IN_COLAB:
        team_tracker = TeamConsistencyTracker(
            history_length=15,  # Longer history for more stability
            confidence_threshold=0.8  # Higher confidence threshold
        )
    else:
        team_tracker = TeamConsistencyTracker()
    
    # Setup annotators
    ellipse_annotator, label_annotator, triangle_annotator = get_annotators()
    
    print("✅ Models and annotators loaded successfully!")
    
    return (
        player_detection_model,
        keypoint_model,
        team_classifier,
        tracker,
        team_tracker,
        ellipse_annotator,
        label_annotator,
        triangle_annotator,
    )

# Load models and annotators
try:
    (
        player_detection_model,

        keypoint_model,    raise

        team_classifier,    print(f"❌ Error loading models: {e}")

        tracker,except Exception as e:

        team_tracker,    

        ellipse_annotator,    ) = setup_models_and_annotators()

        label_annotator,        triangle_annotator,

## Step 6: Process the Video

In [ ]:
# Initialize frame processor with optimized Colab settings
print("Initializing frame processor...")

# Use the configuration from previous cell
USE_STABLE_PROCESSOR = True      # Always use anti-flickering processor

if torch.cuda.is_available():
    print(f"🚀 GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"📊 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

if USE_STABLE_PROCESSOR and IN_COLAB:
    print("Using optimized stable frame processor for Colab...")
    from src.frame_processor_colab import initialize_frame_processor_stable, process_frame_stable
    
    try:\n        initialize_frame_processor_stable(\n            player_detection_model=player_detection_model,\n            keypoint_model=keypoint_model,\n            tracker_obj=tracker,\n            team_classifier_obj=team_classifier,\n            team_tracker_obj=team_tracker,\n            ellipse_ann=ellipse_annotator,\n            label_ann=label_annotator,\n            triangle_ann=triangle_annotator,\n            ball_id=BALL_ID,\n            goalkeeper_id=GOALKEEPER_ID,\n            player_id=PLAYER_ID,\n            referee_id=REFEREE_ID,\n            use_gpu_for_detection=USE_GPU_DETECTION,\n            use_gpu_for_team_classifier=USE_GPU_TEAM_CLASSIFIER,\n        )\n        \n        # Override the process_frame function\n        import src.frame_processor_colab as stable_processor\n        process_frame = stable_processor.process_frame_stable\n        \n        print(\"✅ Optimized frame processor initialized!\")\n        print(f\"   🎯 YOLO Detection: {'GPU' if USE_GPU_DETECTION else 'CPU'}\")\n        print(f\"   🏃 Team Classifier: {'GPU' if USE_GPU_TEAM_CLASSIFIER else 'CPU'}\")\n        \n        if USE_GPU_TEAM_CLASSIFIER:\n            print(\"   ⚡ High-speed mode enabled - team classification every 10 frames\")\n        else:\n            print(\"   🛡️ Stability mode enabled - spatial fallback for reliability\")\n        \n    except Exception as e:\n        print(f\"⚠️ Could not load optimized processor: {e}\")\n        USE_STABLE_PROCESSOR = False\n\nif not USE_STABLE_PROCESSOR or not IN_COLAB:\n    print(\"Using standard frame processor...\")\n    try:\n        initialize_frame_processor(\n            player_detection_model=player_detection_model,\n            keypoint_model=keypoint_model,\n            tracker_obj=tracker,\n            team_classifier_obj=team_classifier,\n            team_tracker_obj=team_tracker,\n            ellipse_ann=ellipse_annotator,\n            label_ann=label_annotator,\n            triangle_ann=triangle_annotator,\n            ball_id=BALL_ID,\n            goalkeeper_id=GOALKEEPER_ID,\n            player_id=PLAYER_ID,\n            referee_id=REFEREE_ID,\n        )\n        \n        print(\"✅ Standard frame processor initialized!\")\n        \n    except Exception as e:\n        print(f\"❌ Error initializing frame processor: {e}\")\n        raise\n\n# Final memory optimization\nif torch.cuda.is_available():\n    torch.cuda.empty_cache()\n    print(\"🧹 GPU cache cleared and ready for processing\")

In [ ]:
def process_video(input_path: str, output_path: Path, max_frames: int = None) -> Path:
    """Process the entire video with anti-flickering improvements"""
    
    print(f"Processing video: {input_path}")
    
    # Get video info
    video_info = sv.VideoInfo.from_video_path(input_path)
    print(
        f"Video info: {video_info.width}x{video_info.height}, {video_info.fps}fps, {video_info.total_frames} frames"
    )
    
    # Limit frames for testing in Colab
    if max_frames and video_info.total_frames > max_frames:
        print(f"⚠️ Limiting processing to {max_frames} frames for Colab performance")
        total_frames_to_process = max_frames
    else:
        total_frames_to_process = video_info.total_frames
    
    # Create output video path
    output_video_path = output_path / f"analyzed_{Path(input_path).name}"
    
    print(f"Output video will be saved to: {output_video_path}")
    
    # PRE-FIT TEAM CLASSIFIER - This is crucial for preventing flickering!
    print("🔄 Pre-analyzing frames for team classification stability...")
    frame_generator_prefit = sv.get_video_frames_generator(input_path)
    initial_crops = []
    prefit_frames = min(30, total_frames_to_process)  # Analyze first 30 frames
    
    for frame_idx, frame in enumerate(frame_generator_prefit):
        if frame_idx >= prefit_frames:
            break
            
        try:
            # Get detections for pre-fitting
            result = player_detection_model.predict(frame, conf=0.3, verbose=False)[0]
            detections = sv.Detections.from_ultralytics(result)
            player_detections = detections[detections.class_id == PLAYER_ID]
            
            if len(player_detections) > 0:
                crops = [sv.crop_image(frame, xyxy) for xyxy in player_detections.xyxy]
                initial_crops.extend(crops[:3])  # Take up to 3 crops per frame
                
            if len(initial_crops) >= 15:  # Stop when we have enough samples
                break
                
        except Exception as e:
            print(f"Warning: Error in pre-analysis frame {frame_idx}: {e}")
            continue
    
    # Fit team classifier if we have enough crops
    if len(initial_crops) >= 8:
        print(f"🎯 Fitting team classifier on {len(initial_crops)} player samples...")
        try:
            # Clear any existing fitted state
            if hasattr(team_classifier, 'reducer'):
                team_classifier.reducer = umap.UMAP(n_components=3, random_state=42)
            if hasattr(team_classifier, 'cluster_model'):
                team_classifier.cluster_model = KMeans(n_clusters=2, random_state=42)
                
            team_classifier.fit(initial_crops)
            print("✅ Team classifier fitted successfully!")
            
            # Verify classifier is properly fitted
            test_prediction = team_classifier.predict(initial_crops[:2])
            print(f"✅ Classifier verification successful: {test_prediction}")
            
        except Exception as e:
            print(f"⚠️ Warning: Could not fit team classifier: {e}")
            print("Will use spatial-based team assignment as fallback")
    else:
        print(f"⚠️ Warning: Only {len(initial_crops)} player samples found (need ≥8)")
        print("Will use spatial-based team assignment")
    
    # Clear memory before main processing
    del frame_generator_prefit
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Process video with improved stability
    print("🎬 Starting main video processing...")
    error_count = 0
    max_errors = 10
    
    with sv.VideoSink(str(output_video_path), video_info) as sink:
        frame_generator = sv.get_video_frames_generator(input_path)
        
        for frame_idx, frame in enumerate(frame_generator):
            if max_frames and frame_idx >= max_frames:
                break
                
            try:
                # Process frame with error handling
                processed_frame = process_frame(frame, frame_idx)
                
                # Ensure frame is valid before writing
                if processed_frame is not None and processed_frame.shape == frame.shape:
                    sink.write_frame(processed_frame)
                else:
                    print(f"⚠️ Warning: Invalid frame at {frame_idx}, using original")
                    sink.write_frame(frame)
                    error_count += 1
                    
            except Exception as e:
                print(f"⚠️ Warning: Error processing frame {frame_idx}: {e}")
                # Use original frame if processing fails
                sink.write_frame(frame)
                error_count += 1
                
                # Stop if too many errors
                if error_count >= max_errors:
                    print(f"❌ Too many errors ({error_count}), stopping processing")
                    break
                continue
            
            # Progress update (less frequent to reduce output)
            if frame_idx % 50 == 0 or frame_idx == total_frames_to_process - 1:
                progress = (frame_idx + 1) / total_frames_to_process * 100
                print(
                    f"Progress: {progress:.1f}% ({frame_idx + 1}/{total_frames_to_process} frames)"
                )
    
    if error_count > 0:
        print(f"⚠️ Processing completed with {error_count} errors")
    
    print(f"\n✅ Video processing completed! Output saved to: {output_video_path}")
    return output_video_path

In [ ]:
# Performance vs Stability Configuration for Colab
print("=== Performance vs Stability Configuration ===")

if torch.cuda.is_available():
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    # Performance strategies
    print(f"🚀 GPU Memory: {gpu_memory_gb:.1f} GB")
    
    # Let user choose strategy
    print("\nAvailable strategies:")
    print("1. 🚀 MAXIMUM SPEED - GPU for everything (3-5x faster, may have stability issues)")
    print("2. ⚡ BALANCED - GPU detection + CPU team classification (2-3x faster, stable)")
    print("3. 🛡️ MAXIMUM STABILITY - CPU for everything (1x speed, most stable)")
    
    # Auto-recommend based on GPU
    if gpu_memory_gb >= 14:
        recommended = 1
        print(f"\n💡 Recommended for your GPU: Strategy 1 (Maximum Speed)")
    elif gpu_memory_gb >= 10:
        recommended = 2
        print(f"\n💡 Recommended for your GPU: Strategy 2 (Balanced)")
    else:
        recommended = 3
        print(f"\n💡 Recommended for your GPU: Strategy 3 (Stability)")
    
    # Configuration options
    STRATEGIES = {
        1: {  # Maximum Speed
            'use_gpu_detection': True,
            'use_gpu_team_classifier': True,
            'max_frames': 500,
            'batch_size': 32,
            'description': 'Fastest processing, may have occasional team flicker'
        },
        2: {  # Balanced 
            'use_gpu_detection': True,
            'use_gpu_team_classifier': False,
            'max_frames': 300,
            'batch_size': 16,
            'description': 'Good speed with stable team assignments'
        },
        3: {  # Maximum Stability
            'use_gpu_detection': False,
            'use_gpu_team_classifier': False,
            'max_frames': 150,
            'batch_size': 8,
            'description': 'Slowest but most stable processing'
        }
    }\n    \n    # Use recommended strategy or let user choose\n    CHOSEN_STRATEGY = recommended  # Change this to 1, 2, or 3 to override\n    \n    config = STRATEGIES[CHOSEN_STRATEGY]\n    print(f\"\\n✅ Using Strategy {CHOSEN_STRATEGY}: {config['description']}\")\n    \nelse:\n    print(\"💻 No GPU available - Using CPU-only processing\")\n    config = {\n        'use_gpu_detection': False,\n        'use_gpu_team_classifier': False,\n        'max_frames': 50,\n        'batch_size': 4,\n        'description': 'CPU-only processing'\n    }\n\n# Apply configuration\nUSE_GPU_DETECTION = config['use_gpu_detection']\nUSE_GPU_TEAM_CLASSIFIER = config['use_gpu_team_classifier']\nMAX_FRAMES = config['max_frames']\n\nprint(f\"\\n📋 Configuration:\")\nprint(f\"  🎯 YOLO Detection: {'GPU' if USE_GPU_DETECTION else 'CPU'}\")\nprint(f\"  🏃 Team Classifier: {'GPU' if USE_GPU_TEAM_CLASSIFIER else 'CPU'}\")\nprint(f\"  📹 Max Frames: {MAX_FRAMES}\")\nprint(f\"  ⚡ Expected Speed: {'3-5x' if config['use_gpu_team_classifier'] else '2-3x' if config['use_gpu_detection'] else '1x'} vs CPU-only\")\n\n# Performance estimates\nif torch.cuda.is_available():\n    if USE_GPU_TEAM_CLASSIFIER:\n        print(f\"  ⏱️ Est. Time (750 frames): ~2-3 minutes\")\n    elif USE_GPU_DETECTION:\n        print(f\"  ⏱️ Est. Time (750 frames): ~4-6 minutes\")  \n    else:\n        print(f\"  ⏱️ Est. Time (750 frames): ~8-12 minutes\")\n\n# Memory management\nif IN_COLAB:\n    import gc\n    import psutil\n    \n    # Check available memory\n    memory = psutil.virtual_memory()\n    print(f\"\\n📊 System Memory:\")\n    print(f\"  Available RAM: {memory.available / 1024**3:.1f} GB\")\n    print(f\"  Memory Usage: {memory.percent:.1f}%\")\n    \n    # Force garbage collection\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    gc.collect()\n\nprint(\"\\n=== Configuration Complete ===\\n\")

## Step 7: Download Results

In [ ]:
# Download the processed video
if IN_COLAB:
    from google.colab import files
    
    try:
        print("Preparing download...")
        files.download(str(output_video_path))
        print("✅ Download initiated! Check your downloads folder.")
    except Exception as e:
        print(f"❌ Download error: {e}")
        print(f"You can manually download the file from: {output_video_path}")
else:
    print(f"✅ Results available at: {output_video_path}")
    print(f"📁 Full output directory: {output_dir}")

## Step 8: Display Video Info and Statistics

In [ ]:
# Display processing statistics
import os

if output_video_path.exists():
    # Get file sizes
    input_size = os.path.getsize(INPUT_VIDEO_PATH) / (1024 * 1024)  # MB
    output_size = os.path.getsize(output_video_path) / (1024 * 1024)  # MB
    
    print("=== Video Processing Statistics ===")
    print(f"📁 Input file size: {input_size:.1f} MB")
    print(f"📁 Output file size: {output_size:.1f} MB")
    print(f"📊 Size change: {((output_size - input_size) / input_size * 100):+.1f}%")
    
    # List all files in output directory
    print(f"\n📂 Output directory contents:")
    for file in output_dir.iterdir():
        if file.is_file():
            size = os.path.getsize(file) / (1024 * 1024)
            print(f"  📄 {file.name} ({size:.1f} MB)")
else:
    print("❌ Output video file not found")

## Additional Configuration Options

You can modify the following variables in the cells above to customize the analysis:

- `MAX_FRAMES`: Limit the number of frames to process (useful for testing)
- `USE_DEFAULT_MODELS`: Set to `True` to use default YOLO models instead of custom ones
- Model parameters in the configuration

### Troubleshooting

1. **Out of memory**: Reduce `MAX_FRAMES` or use CPU instead of GPU
2. **Model not found**: Make sure model files are in the `models/` directory
3. **Import errors**: Ensure all dependencies are installed correctly
4. **Video upload issues**: Make sure the video file is in a supported format (mp4, avi, etc.)

### Performance Tips

1. Use GPU runtime in Colab for better performance
2. Process shorter videos or limit frames for testing
3. Consider reducing video resolution if processing is slow
4. Monitor memory usage in Colab to avoid crashes

---

**Happy analyzing! 🏈⚽**